In [ ]:
# OpenBLAS starts one thread per core even for tiny BLAS-1 calls, and on a machine
# with many cores the thread overhead can dominate them. Pinning it to one thread
# costs nothing here -- the work is in SuperLU, which is single-threaded anyway.
# This must be set *before* numpy is imported: if numpy is already loaded, restart
# the kernel.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import splu
%matplotlib qt5


In [ ]:
# Inputs
# ------

# Immersed body (circular cylinder) and doubly-periodic domain
R  = 1
Lx = 10
Ly = 10
nx = 81
ny = 81

# Timestep. Unlike the steady solver this notebook is derived from, dt is no longer
# an under-relaxation parameter: the geometry advances with it, so it sets the time
# resolution of the body motion and cannot be tuned for convergence. Converging the
# nonlinearity within a step is the job of nOuter instead.
dt     = 0.025
nOuter = 3

# Kinematic viscosity. VAR[IP] is the reduced pressure p/rho, so the density
# never appears explicitly in the equations below.
nu = 1.0

# Net flow rate. The uniform body force in x is adjusted every outer iteration so
# that the bulk velocity sits on this target. In a doubly-periodic box the mean flow
# is otherwise undetermined and drifts, so it has to be pinned one way or the other;
# 0.0 is the natural choice for a body stirring an otherwise quiescent fluid, and it
# is what makes the force on the body meaningful.
uBulkTarget = 0.0

# Wall boundary condition on the immersed body.
#   'noSlip'    u = u_body on the wall: the wall segment carries viscous traction.
#   'freeSlip'  no penetration, zero tangential traction: the wall segment carries
#               none, and that single omission is the only difference between them.
#
# Measured order on a CURVED wall against exact solutions (Poisson on a disk,
# N = 20...320):
#   freeSlip   second order   (L2 orders 2.21, 2.33, 2.06, 2.14)
#   noSlip     first  order   (L2 orders 0.75, 1.27, 0.91, 0.90)
# Free slip needs no wall-normal distance at all -- theta and alpha are exact -- so
# nothing degrades it. No slip needs one, estimated as alpha*V/(2*A_wall), and that
# estimate is what costs the order.
bcType = 'noSlip'

# Body motion
# -----------
# The path is prescribed. bodyVelocity must be the exact derivative of bodyPosition:
# the first supplies the wall condition, the second the mass source, and if they
# disagree the two halves of the boundary condition pull against each other.
#   'still'      the stationary body of the parent notebook
#   'translate'  uniform translation, wrapping through the periodic seam
#   'oscillate'  harmonic motion along x
motion   = 'oscillate'
uTrans   = 0.5             # 'translate': speed
aOsc     = 1.0             # 'oscillate': amplitude
fOsc     = 0.05            # 'oscillate': frequency
xc0, yc0 = Lx/2, Ly/2      # body centre at t = 0


def bodyPosition(t):
    if motion == 'still':      return xc0, yc0
    if motion == 'translate':  return xc0+uTrans*t, yc0
    if motion == 'oscillate':  return xc0+aOsc*np.sin(2*np.pi*fOsc*t), yc0
    raise ValueError(f'unknown motion {motion!r}')


def bodyVelocity(t):
    if motion == 'still':      return np.array([0.0, 0.0])
    if motion == 'translate':  return np.array([uTrans, 0.0])
    if motion == 'oscillate':
        return np.array([2*np.pi*fOsc*aOsc*np.cos(2*np.pi*fOsc*t), 0.0])
    raise ValueError(f'unknown motion {motion!r}')

In [ ]:
# Grid, fields and sparsity pattern
# ---------------------------------
IU, IV, IP = 0, 1, 2                 # component indices into VAR

dx, dy = Lx/nx, Ly/ny
V      = dx*dy                       # cell volume
nCells = nx*ny

xp = (np.arange(nx)+0.5)*dx          # cell centres
yp = (np.arange(ny)+0.5)*dy
xf = np.arange(nx+1)*dx              # face centres
yf = np.arange(ny+1)*dy
XP, YP = np.meshgrid(xp, yp)
XF, YF = np.meshgrid(xf, yf)

VAR = np.zeros((3, ny, nx))          # u, v, p

# The 5-point stencil is held as five (ny,nx) coefficient arrays in the order
# P, E, W, N, S -- not as an (nCells,nCells) matrix. Its sparsity pattern never
# changes, so the CSR structure and the permutation from stencil order into CSR
# data order are built once, here. Assembling a matrix is then an O(nnz)
# fancy-index rather than an O(nCells^2) scan over a 99.92%-empty dense array.
ID  = np.arange(nCells).reshape(ny, nx)
ROW = np.stack([ID]*5, -1).ravel()
COL = np.stack([ID, np.roll(ID, -1, 1), np.roll(ID, 1, 1),
                    np.roll(ID, -1, 0), np.roll(ID, 1, 0)], -1).ravel()
assert len(set(zip(ROW.tolist(), COL.tolist()))) == ROW.size, \
    'stencil aliases itself: nx and ny must both be at least 3'

_pattern    = csr_matrix((np.arange(1.0, ROW.size+1), (ROW, COL)),
                         shape=(nCells, nCells))
CSR_PERM    = (_pattern.data-1).astype(np.int64)
CSR_INDICES = _pattern.indices
CSR_INDPTR  = _pattern.indptr


def zeroStencil():
    """An empty set of stencil coefficients, indexed [P|E|W|N|S, iy, ix]."""
    return np.zeros((5, ny, nx))


def toCSR(a):
    """Assemble stencil coefficients into a sparse matrix."""
    return csr_matrix((a.reshape(5, -1).T.ravel()[CSR_PERM], CSR_INDICES, CSR_INDPTR),
                      shape=(nCells, nCells))


def offDiagDot(a, phi):
    """(A - diag(aP)) @ phi, as four neighbour shifts. No matrix is formed."""
    return (a[1]*np.roll(phi, -1, 1) + a[2]*np.roll(phi, 1, 1)
          + a[3]*np.roll(phi, -1, 0) + a[4]*np.roll(phi, 1, 0))


def wrapX(f):
    """Cell-array -> face-array in x: append the periodic image of the first face."""
    return np.concatenate([f, f[:, :1]], axis=1)


def wrapY(f):
    return np.concatenate([f, f[:1, :]], axis=0)

## Cut-cell immersed boundary on a **moving** body — no slip *and* free slip

The stationary solver of `SIMPLEC_faceArea_noSlip.ipynb`, extended to a body that moves through
the mesh. Set `bcType` to `'noSlip'` or `'freeSlip'` and `motion` to `'still'`, `'translate'`
or `'oscillate'`. Full description: `doc/moving_ibm.pdf`; the stationary formulation is in
`doc/freeslip_ibm_part2.pdf`.

**The mesh never moves.** Only $\theta$, $\alpha$ and the wall segment depend on time, so this
is not an ALE method: no axis face ever moves, and no mesh-velocity term appears anywhere.

### What changes

Applying Reynolds transport to the fluid part of a cell — bounded by the wet parts of the
*fixed* axis faces plus the *moving* wall segment, the former sweeping no volume — gives

$$\frac{\mathrm{d}(\alpha_P V)}{\mathrm{d}t}=(\mathbf{u}_b\!\cdot\!\mathbf{n}_{\rm wall})A_{\rm wall},
\qquad\text{hence}\qquad
\sum_f\theta_f\phi_f=-(\mathbf{u}_b\!\cdot\!\mathbf{n}_{\rm wall})A_{\rm wall}=\mathbf{u}_b\!\cdot\!\mathbf{S}_w .$$

Continuity is the **only** equation that changes. Every other term keeps its static form,
because the wall is impermeable in its own frame and the relative flux
$(\mathbf{u}-\mathbf{u}_b)\!\cdot\!\mathbf{n}$ through it is still zero.

| term | body at rest | body moving |
|---|---|---|
| continuity | $\sum_f\theta_f\phi_f=0$ | $\sum_f\theta_f\phi_f=\mathbf{u}_b\!\cdot\!\mathbf{S}_w$ |
| convective | no wall term | no wall term (relative flux still zero) |
| pressure | $p_P\mathbf{n}_{\rm wall}A_{\rm wall}$, absorbed by closure | same |
| free slip | $\mathbf{u}\!\cdot\!\mathbf{n}=0$ *emerges* from closure | $\mathbf{u}\!\cdot\!\mathbf{n}=\mathbf{u}_b\!\cdot\!\mathbf{n}$ emerges from the same argument |
| no slip | $\nu(0-u_P)A_{\rm wall}/d$ | $\nu(u_b-u_P)A_{\rm wall}/d$: same diagonal, new source |
| $\partial/\partial t$ | $\alpha V(u^{n+1}\!-u^n)/\Delta t$ | $(\alpha^{n+1}u^{n+1}-\alpha^{n}u^{n})V/\Delta t$ |

**The mass source *is* the boundary condition.** For free slip nothing else is added at all: a
uniform $\mathbf{c}$ in a cut cell gives $\sum_f\theta_f\mathbf{c}\!\cdot\!\mathbf{n}_fA_f=
\mathbf{c}\!\cdot\!\mathbf{S}_w$ by closure, and setting that equal to the source forces
$\mathbf{c}\!\cdot\!\mathbf{S}_w=\mathbf{u}_b\!\cdot\!\mathbf{S}_w$.

### Which quantity is primary

The source is taken from the **flux**, $S=-\mathbf{u}_b\!\cdot\!\mathbf{S}_w$, and the volume
change from the source,

$$\alpha^{n}V:=\alpha^{n+1}V-\Delta t\,S ,$$

not the other way round. Both directions look equivalent on paper; only this one works.

* $\mathbf{S}_w$ is built from the same $\theta$ as `interpolatePhi`, so a uniform
  $\mathbf{u}=\mathbf{u}_b$ gives $\operatorname{div}\phi=\mathbf{u}_b\!\cdot\!\mathbf{S}_w$
  *identically*, and $\sum_P S_P$ telescopes to zero to roundoff. The pressure reference is
  imposed by doubling one diagonal (`fvMatrix::setReference`), which returns the true solution
  only for a compatible right-hand side; here nothing has to be projected away.
* Defining $\alpha^n$ by integrating $S$ makes the **discrete geometric conservation law** hold
  by construction: with $\mathbf{u}\equiv\mathbf{u}_b$ the unsteady term
  $(\alpha^{n+1}-\alpha^n)V\mathbf{u}_b/\Delta t=S\,\mathbf{u}_b$ is cancelled *identically* by
  the convective term $\mathbf{u}_b\sum_f\theta_f\phi_f=-S\,\mathbf{u}_b$.
* The reverse pairing — a geometric $(\alpha^{n+1}-\alpha^{n})V/\Delta t$ driving the source —
  fails for a concrete reason. `cellFraction` samples $\alpha$ to about $10^{-3}$ per cell, and
  dividing that noise by $\Delta t$ amplifies it without bound: measured against
  $-\mathbf{u}_b\!\cdot\!\mathbf{S}_w$, the difference does not shrink as $\Delta t\to0$, and the
  free-stream error is 50% of $u_b$ instead of $10^{-15}$.

$\alpha^n$ is recomputed from the current geometry every step, so it is a per-step quantity and
cannot drift.

### Verification

**Free stream.** Fill the domain with $\mathbf{u}=\mathbf{u}_b$ and translate the body at that
velocity: the fluid moves rigidly with the body, so $\mathbf{u}\equiv\mathbf{u}_b$ is the exact
solution and any departure is a discretisation error. It tests every sign at once.

| $u_b$ | $n_{\rm outer}=1$ | 2 | 3 | 5 |
|---|---|---|---|---|
| 0.37 | 5.0e-16 | 5.6e-16 | 5.6e-16 | 5.6e-16 |
| 2.00 | 3.6e-15 | 3.1e-15 | 3.6e-15 | 3.6e-15 |

Machine precision at any speed and from the first outer iteration. That last part needs one
extra line: the flux carried across the step is re-interpolated onto the new $\theta$ before the
outer loop starts. Without it a face that has just opened carries no history and one that has
just closed carries a history that no longer exists, and the error is $10^{-3}\!-\!10^{-1}$
instead — converging only as $\rho^{\,n_{\rm outer}}$ with $\rho\approx0.85$.

**Mass.** $\max|\operatorname{div}\phi+S|\sim10^{-16}$ and $\sum(\operatorname{div}\phi+S)\sim
10^{-17}$ every step, with no projection anywhere, including across sliver absorption events.

**Galilean invariance.** A fixed body in a bulk flow $+U$ and a body translating at $+U$ through
a fluid of zero bulk velocity must give opposite forces, and are computed through entirely
different terms. Measured at $U=0.5$, $\nu=1$, $n_x=81$: $F_x=+5.66034$ fixed against
$F_x=-5.69040$ moving, **0.53%** apart — against a $\pm2.8\%$ ripple as the body crosses cells,
which is the familiar cut-cell signature and an order of magnitude larger than the discrepancy
in the mean.

### Step restriction

$\alpha^{n}$ goes negative — and the row sum of the unsteady plus convective operator, which is
exactly $\alpha^{n}V/\Delta t$, changes sign with it — once the wall sweeps through more than a
wall-adjacent cell holds:

$$\Delta t\,|\mathbf{u}_b| \;\lesssim\; \frac{\alpha_P V}{A_{\rm wall}} \;=\; 2d .$$

This is a condition on the **smallest wet cell**, not on the mesh spacing, so `ALPHA_MIN` is what
sets it. The run loop reports the smallest $\alpha^n$ and how many cells violate it; a handful of
cells is harmless (the diagonal is still dominated by the viscous and wall terms), a large count
means the step is too big.

### Limitations

* **First order in time** (implicit Euler), and first order in space for no slip — the
  wall-normal distance $d=\alpha_PV/(2A_{\rm wall})$ is what costs the spatial order. Free slip
  is second order in space.
* **Absorbed slivers lose their content.** A cell that crosses `ALPHA_MIN` hands over no
  momentum. The effect is $O(\texttt{ALPHA\_MIN}\cdot V)$ and is the same defect the stationary
  solver has; cell *merging* rather than absorption is the clean fix. Note that mass
  conservation is unaffected: an absorbed cell has all faces closed, so $\mathbf{S}_w=0$ and it
  contributes nothing to the source.
* **Prescribed motion only.** `bodyForce` returns the hydrodynamic force, so a rigid-body ODE
  would close the loop, but a light body would need the added-mass sub-iteration inside the
  outer loop.

In [ ]:
# Finite-volume operators on the cut mesh
# ---------------------------------------
# Every face carries its fluid area fraction theta, every cell its fluid volume
# fraction alpha. Both are now functions of time -- the mesh never moves, only the
# fractions do. The fluid part of a cut cell is a polygon whose boundary is the
# partially-open axis faces plus the wall segment, and geometric closure
#
#     sum_f theta_f n_f A_f + n_wall A_wall = 0                              (*)
#
# holds to machine precision because theta and the wall segment come from the same
# level set. That single identity carries the whole free-slip boundary condition:
#
#   * viscous   -- free slip means zero tangential stress on the wall segment, and
#                  at first order no normal viscous stress either, so the wall
#                  contributes nothing and the operator is just the theta-weighted
#                  face sum. It annihilates a constant field, as it must.
#   * convective-- the wall is impermeable in its own frame, so the RELATIVE flux
#                  (u - u_b).n through it vanishes and the wall contributes nothing
#                  here either. The axis faces do not move and therefore carry the
#                  absolute flux, exactly as in the static case: no mesh-velocity
#                  term appears anywhere.
#   * pressure  -- the wall DOES carry a pressure force (this is the form drag).
#                  With p_wall = p_P to first order, (*) turns the Gauss gradient
#                  into  sum_f theta_f (p_f - p_P) n_f A_f, which vanishes for a
#                  uniform pressure, as it must.
#   * continuity-- sum_f theta_f phi_f = u_b . Sw, the discrete wall flux. This is
#                  the ONE equation a moving wall changes. For a body at rest the
#                  right-hand side is zero and no-penetration EMERGES from continuity
#                  plus closure; for a moving one the same argument gives
#                  u.n = u_b.n. The mass source IS the moving-wall condition.
#
# So there is still no ghost value, no blanking of cut cells, and no cross term
# coupling the velocity components.


def fvm_ddt_LHS(a, alphaVNew, dt):
    """Implicit half of d/dt(alpha V u), evaluated at the NEW fluid volume."""
    a[0] += alphaVNew/dt


def fvm_ddt_RHS(rhs, phi, alphaVOld, dt):
    """
    Explicit half, at the OLD fluid volume. Carrying two volumes rather than one is
    what makes the pair conservative across a change of geometry: a cell that has
    just emerged from the body has alphaVOld = 0 and so needs no history, and a cell
    about to be swallowed hands its momentum on rather than keeping it.
    """
    rhs += alphaVOld*phi/dt


def fvm_lap(a, GammaX, GammaY=None):
    """theta-weighted implicit laplacian; no wall term (zero tangential stress)."""
    if GammaY is None:
        GammaY = GammaX
    gE = 0.5*(GammaX+np.roll(GammaX, -1, 1))*(dy/dx)*thX[:, 1:]
    gW = 0.5*(GammaX+np.roll(GammaX,  1, 1))*(dy/dx)*thX[:, :-1]
    gN = 0.5*(GammaY+np.roll(GammaY, -1, 0))*(dx/dy)*thY[1:, :]
    gS = 0.5*(GammaY+np.roll(GammaY,  1, 0))*(dx/dy)*thY[:-1, :]
    a[0] -= gE+gW+gN+gS
    a[1] += gE
    a[2] += gW
    a[3] += gN
    a[4] += gS


def fvm_divConvFlux(a, phiX, phiY):
    """theta-weighted central div(phi u); phi already carries the wet face area."""
    fE, fW = phiX[:, 1:], phiX[:, :-1]
    fN, fS = phiY[1:, :], phiY[:-1, :]
    a[0] += 0.5*(fE-fW+fN-fS)
    a[1] += 0.5*fE
    a[2] -= 0.5*fW
    a[3] += 0.5*fN
    a[4] -= 0.5*fS


def fvc_grd(phi):
    """
    Volume-integrated Gauss gradient including the wall pressure force, written via
    the closure identity as sum_f theta_f (p_f - p_P) n_f A_f. Exactly zero for a
    uniform pressure, so no spurious force is generated on a cut cell.
    """
    dE = 0.5*(np.roll(phi, -1, 1)-phi)*thX[:, 1:]
    dW = 0.5*(np.roll(phi,  1, 1)-phi)*thX[:, :-1]
    dN = 0.5*(np.roll(phi, -1, 0)-phi)*thY[1:, :]
    dS = 0.5*(np.roll(phi,  1, 0)-phi)*thY[:-1, :]
    return np.stack([dy*(dE-dW), dx*(dN-dS)])


def fvc_grdFlux(phi, GammaX, GammaY=None):
    """Face fluxes of fvm_lap: theta-weighted, and matching it coefficient for coefficient."""
    if GammaY is None:
        GammaY = GammaX
    fx = (phi-np.roll(phi, 1, 1))/dx*0.5*(GammaX+np.roll(GammaX, 1, 1))*dy
    fy = (phi-np.roll(phi, 1, 0))/dy*0.5*(GammaY+np.roll(GammaY, 1, 0))*dx
    return wrapX(fx)*thX, wrapY(fy)*thY


def fvc_divFlux(phiX, phiY):
    return (phiX[:, 1:]-phiX[:, :-1]) + (phiY[1:, :]-phiY[:-1, :])


def interpolatePhi(Ux, Uy):
    """Face volume fluxes through the wet part of each face."""
    return (0.5*(wrapX(np.roll(Ux, 1, 1))+wrapX(Ux))*dy*thX,
            0.5*(wrapY(np.roll(Uy, 1, 0))+wrapY(Uy))*dx*thY)

In [ ]:
# Immersed boundary: cut cells on a moving body
# ---------------------------------------------
# The mesh is fixed and the body moves through it, so only theta, alpha and the wall
# segment depend on time. This is not an ALE method: no axis face ever moves, and
# the only trace of the motion in the discretisation is what the wall segment does.

def levelSet(x, y, xc, yc):
    """
    Signed distance to the surface of a body centred at (xc, yc), negative inside.
    Distances are taken in the minimum image, so the body may cross the periodic seam
    and xc may run off the end of the domain.

    That it is a signed DISTANCE, and not merely some function with the right zero
    set, is used by the band tests below: they decide from the value at a single
    point whether a face or a cell can possibly be cut.
    """
    dxb = (x-xc+Lx/2) % Lx - Lx/2
    dyb = (y-yc+Ly/2) % Ly - Ly/2
    return np.hypot(dxb, dyb) - R


ALPHA_MIN = 0.02   # cells with less fluid than this are absorbed into the body,
                   # which moves the boundary by O(ALPHA_MIN*h) and keeps the
                   # small-cell problem out of the linear systems

IBM_SOLID_WEIGHT = 1e12

# Cut-cell geometry: face fluid fractions and cell fluid fractions
# ---------------------------------------------------------------
# theta_f in [0,1] is the fraction of face f that lies in the fluid; alpha is the
# same for the cell volume. They replace the boolean open/closed mask: the body is
# opaque again (no flux crosses the solid), but a face that the boundary cuts stays
# partly open, so tangential flow is not blocked.
#
# The property that makes this work is geometric closure. For the fluid polygon of
# a cut cell,
#
#     sum_f theta_f n_f A_f  +  n_wall A_wall = 0 ,
#
# so for a uniform field parallel to the wall the discrete divergence reduces to
# -(u.n_wall) A_wall. No ghost is needed for that, and it holds however oblique the
# wall is -- and, now, however fast it moves.
#
# All of it is recomputed every timestep, so both routines work on a band: a face or
# a cell more than half a diagonal from the surface cannot be cut, and the signed
# distance settles it without any quadrature.

def segmentFluidFraction(x0, y0, x1, y1, xc, yc, nBisect=40):
    """
    Fluid fraction of the straight segment from (x0,y0) to (x1,y1), by bisection on
    the parametrised segment. Only the segments the surface actually crosses are
    bisected. Assumes at most one crossing per segment, which holds once the body is
    resolved.
    """
    f0 = levelSet(x0, y0, xc, yc)
    f1 = levelSet(x1, y1, xc, yc)
    frac = (f0 > 0.0).astype(float)
    cut  = (f0 > 0.0) != (f1 > 0.0)
    if cut.any():
        xa, ya = x0[cut], y0[cut]
        sx, sy = (x1-x0)[cut], (y1-y0)[cut]
        fa = f0[cut]
        lo = np.zeros_like(fa)
        hi = np.ones_like(fa)
        for _ in range(nBisect):
            mid = 0.5*(lo+hi)
            sameAsA = (levelSet(xa+mid*sx, ya+mid*sy, xc, yc) > 0.0) == (fa > 0.0)
            lo = np.where(sameAsA, mid, lo)
            hi = np.where(sameAsA, hi, mid)
        tCross = 0.5*(lo+hi)
        frac[cut] = np.clip(np.where(fa > 0.0, tCross, 1.0-tCross), 0.0, 1.0)
    return frac


def faceFractions(xc, yc):
    """theta on the x-faces (ny, nx+1) and the y-faces (ny+1, nx)."""
    XF1 = np.tile(xf, (ny, 1))
    Y0  = np.tile(yf[:-1][:, None], (1, nx+1))
    thX = segmentFluidFraction(XF1, Y0, XF1, Y0+dy, xc, yc)
    YF1 = np.tile(yf[:, None], (1, nx))
    X0  = np.tile(xf[:-1], (ny+1, 1))
    thY = segmentFluidFraction(X0, YF1, X0+dx, YF1, xc, yc)
    # x=0 and x=Lx are the same periodic face; likewise y. Compute once, copy, so
    # that conservation cannot be broken by an inconsistent seam.
    thX[:, -1] = thX[:, 0]
    thY[-1, :] = thY[0, :]
    return thX, thY


def cellFraction(xc, yc, nSub=32):
    """alpha, the fluid fraction of each cell, by sub-sampling the band of cut cells."""
    sd  = levelSet(XP, YP, xc, yc)
    cut = np.abs(sd) < 0.5*np.hypot(dx, dy)
    a   = (sd > 0.0).astype(float)
    if cut.any():
        x0, y0 = XF[:-1, :-1][cut], YF[:-1, :-1][cut]
        s   = (np.arange(nSub)+0.5)/nSub
        acc = np.zeros(x0.shape)
        for sx in s:
            for sy in s:
                acc += levelSet(x0+sx*dx, y0+sy*dy, xc, yc) > 0.0
        a[cut] = acc/(nSub*nSub)
    return a


def updateGeometry(xc, yc):
    """
    Move the body to (xc, yc) and rebuild everything that depends on where it is.
    All of it is module-level, exactly as in the static notebook, so the operators
    above are untouched by the motion.
    """
    global thX, thY, alpha, alphaV, fluid, solid, nearBody, sdCell
    global Swx, Swy, Awall, dWall

    thX, thY = faceFractions(xc, yc)
    alpha    = cellFraction(xc, yc)
    sdCell   = levelSet(XP, YP, xc, yc)

    # Absorb slivers: a deactivated cell must also have its faces closed, or the
    # discrete divergence would see a flux into a cell that carries no equation.
    tiny  = alpha < ALPHA_MIN
    alpha = np.where(tiny, 0.0, alpha)
    thX[:, :-1] = np.where(tiny, 0.0, thX[:, :-1]); thX[:, 1:] = np.where(tiny, 0.0, thX[:, 1:])
    thY[:-1, :] = np.where(tiny, 0.0, thY[:-1, :]); thY[1:, :] = np.where(tiny, 0.0, thY[1:, :])

    fluid    = alpha > 0.0
    solid    = ~fluid
    alphaV   = alpha*V
    nearBody = fluid & (np.roll(solid, 1, 1) | np.roll(solid, -1, 1) |
                        np.roll(solid, 1, 0) | np.roll(solid, -1, 0))

    # Wall segment, recovered from the closure identity rather than computed
    # separately, so that it is consistent with the face fractions by construction:
    #     n_wall A_wall = - sum_f theta_f n_f A_f .
    Swx   = (thX[:, 1:]-thX[:, :-1])*dy
    Swy   = (thY[1:, :]-thY[:-1, :])*dx
    Awall = np.hypot(Swx, Swy)

    # Effective wall-normal distance from the cell average to the wall. A wall-adjacent
    # cell holds fluid of volume alpha*V behind a wall segment of area A_wall, so its
    # mean thickness is alpha*V/A_wall; for a profile that is linear in the wall-normal
    # direction the cell average sits at half of that.
    dWall = np.where(Awall > 0.0, alphaV/(2.0*np.maximum(Awall, 1e-300)), np.inf)


def ibm_noSlipWall(a, Gamma):
    """
    Implicit half of the viscous traction of a no-slip wall segment.

    Free slip leaves the wall segment stress-free, which is why it contributes
    nothing there. No slip does not: with the wall moving at u_b it carries

        int_wall Gamma du/dn dS  ~  Gamma (u_b - u_P)/dWall * A_wall ,

    whose u_P half is this single diagonal term -- unchanged from the static case,
    because the body velocity does not enter it. It is negative in the laplacian and
    therefore adds to the momentum diagonal once the laplacian is subtracted, and it
    grows like 1/alpha as a cell is squeezed against the wall: the closer the
    boundary comes to a cell, the more diagonally dominant the system becomes, and
    u_P is driven to u_b rather than the matrix degrading.
    """
    a[0] -= np.where(Awall > 0.0, Gamma*Awall/dWall, 0.0)


def ibm_wallSource(Gamma, uWallComponent):
    """
    The u_b half of the same traction, which is a source rather than a coefficient.
    It is the whole of what a moving no-slip wall adds to the momentum equation, and
    it vanishes for a wall at rest -- which is why the static formulation needs no
    source at all.
    """
    return np.where(Awall > 0.0, Gamma*Awall/dWall, 0.0)*uWallComponent


def ibm_blankSolid(a):
    """Cells with no fluid carry no equation."""
    a[0] += np.where(solid, IBM_SOLID_WEIGHT, 0.0)


def ibm_blankSource(uWallComponent):
    """
    Drives the blanked interior to the body velocity instead of to zero. No fluid
    cell ever reads those values -- every face of a solid cell is closed -- but a
    cell emerging from the body then starts from something sensible, and the plots
    show the body moving rather than a hole.
    """
    return np.where(solid, IBM_SOLID_WEIGHT, 0.0)*uWallComponent


def ibm_movingWall(uWall, dt):
    """
    Everything a moving wall adds to the discretisation: the mass source of the
    continuity equation, and the fluid volume the cells held at the start of the step.

    Continuity over the fluid part of a cell, whose boundary is the wet parts of the
    FIXED axis faces plus the MOVING wall segment, reads

        sum_f theta_f phi_f  =  -(u_b.n_wall) A_wall  =  u_b . Sw ,

    the axis faces sweeping no volume because they lie in fixed planes, and Sw being
    the closure vector -n_wall A_wall = sum_f theta_f n_f A_f already assembled by
    updateGeometry. The projection step is made to deliver div(phi) = -S with

        S = -(u_b . Sw) ,

    and continuity is the ONLY equation a moving wall changes.

    Two properties come free with this form, and neither survives if the source is
    instead taken as the geometric volume difference (alpha^{n+1}-alpha^n)V/dt:

      * it is EXACTLY compatible. Sw telescopes to zero over the periodic domain, so
        sum(S) vanishes to roundoff, for every body position and every sliver pattern.
        That matters: the pressure system is a pure Neumann problem on the fluid whose
        reference is imposed by doubling one diagonal (fvMatrix::setReference), and
        that returns the true solution only for a compatible right-hand side --
        otherwise the whole mismatch is deposited in the reference cell as a spurious
        point source. Here nothing has to be projected away.
      * it is exact for the divergence operator it is paired with. Sw is built from
        the same theta as interpolatePhi, so a uniform u = u_b gives div(phi) = u_b.Sw
        identically, and no-penetration on a moving wall holds to machine precision
        rather than to the accuracy of a volume quadrature.

    The old fluid volume is then obtained by INTEGRATING that same source rather than
    by differencing the geometry:

        alpha^n V = alpha^{n+1} V - dt S .

    This is the discrete geometric conservation law, and it is what makes the free
    stream exact: put u = u_b uniform, and the unsteady term
    (alpha^{n+1}-alpha^n)V u_b/dt = S u_b is cancelled identically by the convective
    term u_b sum_f theta_f phi_f = -S u_b, at any dt and along any body path. The
    reverse pairing -- a geometric volume difference driving the mass source -- fails
    for a concrete reason: cellFraction samples alpha to about 1e-3 per cell, and
    dividing that noise by dt amplifies it without bound as the step is refined.

    Note that alpha^n is recomputed from the current geometry every step, so it is a
    per-step quantity and not an integrated state: it cannot drift. Where it goes
    negative the body has moved more than a cell in one step, and the row sum of the
    unsteady plus convective operator, which is exactly alpha^n V/dt, changes sign.
    That is the body Courant condition, and the run loop reports it.
    """
    S = -(uWall[0]*Swx + uWall[1]*Swy)
    return S, alphaV - dt*S

In [ ]:
# Run controls
# ------------
startNew    = True
readRestart = False
consistent  = True          # True: SIMPLEC, False: SIMPLE

nSteps = 400          # dt = 0.025, so t = 10: half a period of the default motion

# There is no adaptive step control and no rollback here. Both belonged to the steady
# solver, where dt was only a relaxation parameter; rolling a step back now would
# leave the geometry at a time the solution is not at. The Courant number printed
# below is a diagnostic, not a constraint the code enforces -- if it is large, lower
# dt or raise nOuter.


def bulkVelocity(u):
    return np.sum(u*alphaV)/np.sum(alphaV)


def referenceCell():
    """
    Pressure reference: the fluid cell farthest from the body, so that the body never
    swallows it as it moves. With every cut face closed the fluid region is a
    pure-Neumann island, so it needs exactly one reference -- and would need one per
    region if the body ever split the fluid in two.
    """
    return divmod(int(np.argmax(np.where(fluid, sdCell, -np.inf))), nx)


def bodyForce(p, U, uWall):
    """
    Hydrodynamic force per unit density on the body. By the closure identity the
    outward normal of the BODY times its area is (Swx, Swy), so the pressure force is
    -sum p_P (Swx, Swy); the viscous force is the reaction to the wall traction that
    ibm_noSlipWall and ibm_wallSource already assemble, and is zero for free slip.
    """
    fPres = -np.array([np.sum(p*Swx), np.sum(p*Swy)])
    if bcType == 'noSlip':
        c = np.where(Awall > 0.0, nu*Awall/dWall, 0.0)
        fVisc = np.array([np.sum(c*(U[iC]-uWall[iC])) for iC in range(2)])
    else:
        fVisc = np.zeros(2)
    return fPres+fVisc


def timeStep(t, dt, VAR, phiHbyAx, phiHbyAy):
    """
    One physical timestep: move the geometry to t+dt, then run nOuter SIMPLEC
    iterations on it while it is held fixed.

    The outer loop is what pimpleFoam adds to simpleFoam and what a moving body
    needs. With a single pass the convective flux would be assembled from a geometry
    one body-displacement out of date, and the discrete conservation law below would
    hold only to O(dt) instead of exactly.

    VAR is updated in place; the face fluxes and a diagnostics dictionary are
    returned.
    """
    VARold = VAR.copy()

    # Geometry at the new time level. The old geometry is never needed: the volume
    # the cells had at the start of the step is recovered from the wall flux itself,
    # which is what makes the discrete conservation law exact.
    updateGeometry(*bodyPosition(t+dt))
    uWall = bodyVelocity(t+dt)
    massSrc, alphaVOld = ibm_movingWall(uWall, dt)
    jRef, iRef = referenceCell()

    # Put the carried flux back on the geometry the step is about to use. phiHbyA
    # came off the faces of the OLD cut mesh: a face that has just opened has no
    # history there at all, and one that has just closed carries a history that no
    # longer exists. Re-interpolating u^n through the new theta costs one line and
    # starts the outer loop on the right mesh -- for a fluid translating rigidly with
    # the body it starts it exactly ON the fixed point, which is what makes the free
    # stream preserved to machine precision even at nOuter = 1. Nothing is lost by
    # discarding the old flux: this is only the initial guess for the convective
    # operator, and the flux the step returns is the Rhie-Chow projected one either way.
    phiHbyAx, phiHbyAy = interpolatePhi(VARold[IU], VARold[IV])

    # Sources, all fixed for the whole step: only the pressure and the convective
    # flux change from one outer iteration to the next.
    rhs = np.zeros((2, ny, nx))
    for iC in range(2):
        fvm_ddt_RHS(rhs[iC], VARold[iC], alphaVOld, dt)
        rhs[iC] += ibm_blankSource(uWall[iC])
        if bcType == 'noSlip':
            rhs[iC] += ibm_wallSource(nu, uWall[iC])
    unitSource = np.stack([alphaV, np.zeros((ny, nx))])   # source for gx = 1
    noField    = np.zeros((ny, nx))

    # Static part of the momentum operator at this geometry. Free slip needs one
    # laplacian per velocity component; no slip adds the wall traction to it.
    lapl = zeroStencil()
    fvm_lap(lapl, nu*np.ones((ny, nx)))
    if bcType == 'noSlip':
        ibm_noSlipWall(lapl, nu)
    elif bcType != 'freeSlip':
        raise ValueError("bcType must be 'noSlip' or 'freeSlip'")

    for outer in range(nOuter):

        aMom = []
        for iC in range(2):
            a = zeroStencil()
            fvm_ddt_LHS(a, alphaV, dt)                    # d/dt, at the new volume
            a -= lapl                                     # -laplacian(nu, u), IBM-corrected
            aMom.append(a)
        for a in aMom:
            fvm_divConvFlux(a, phiHbyAx, phiHbyAy)
        for a in aMom:
            ibm_blankSolid(a)                             # u = u_body inside the body

        # Split as  aP*u = H(u) - V*grad(p),  H(u) = source - offDiag*u
        aP     = np.stack([a[0] for a in aMom])                # OpenFOAM A(), times V
        sumOff = np.stack([a[1:].sum(axis=0) for a in aMom])   # OpenFOAM -H1(), times V
        # S1: keep the SIMPLEC mobilities positive. On a cut arm the upwind inflow
        # term has no off-diagonal to land on, so it falls on the diagonal and can
        # push aP (and hence aP+sumOff) non-positive on a few cells, which makes
        # rAtU negative and turns the pressure laplacian indefinite. Floor both at
        # the unsteady value alpha V/dt, which is what an unobstructed cell has.
        aP    = np.maximum(aP, alphaV/dt)
        rAU   = 1.0/aP
        rAtU  = 1.0/np.maximum(aP+sumOff, alphaV/dt) if consistent else rAU
        rAtUV = rAtU*V

        # Pressure matrix. rAtU differs between u and v under free slip, so x-faces
        # use the u mobility and y-faces the v one. (OpenFOAM instead averages the
        # components in A(); see addCmptAvBoundaryDiag.)
        pStencil = zeroStencil()
        fvm_lap(pStencil, rAtUV[IU], rAtUV[IV])
        # Every face of a solid cell is closed, so its row came out empty: give it an
        # identity row, with the sign of the negative-definite laplacian.
        pStencil[0][solid] = -1.0
        pStencil[0, jRef, iRef] *= 2.0                    # OpenFOAM fvMatrix::setReference:
        Ap = toCSR(pStencil)                              # symmetric, unlike replacing the row

        # Factorise once; both passes below reuse it. The cut-cell boundary
        # introduces no coupling between the velocity components, so the momentum
        # equations are two independent scalar problems.
        luMom = [splu(toCSR(a).tocsc()) for a in aMom]
        luP   = splu(Ap.tocsc())

        def advance(sourceField, pOld, massSource):
            """
            One predictor-projection pass with the operators above. The whole pass is
            affine in (sourceField, pOld, massSource), which is what the flow-rate
            constraint exploits.
            """
            RHS   = sourceField.reshape(2, -1)
            gradP = fvc_grd(pOld)
            U = np.stack([luMom[iC].solve(RHS[iC]-gradP[iC].ravel()).reshape(ny, nx)
                          for iC in range(2)])
            HbyA = np.stack([(sourceField[iC] - offDiagDot(aMom[iC], U[iC]))*rAU[iC]
                             for iC in range(2)])
            pX, pY = interpolatePhi(HbyA[IU], HbyA[IV])
            if consistent:
                fX, fY = fvc_grdFlux(pOld, (rAtU[IU]-rAU[IU])*V, (rAtU[IV]-rAU[IV])*V)
                pX, pY = pX+fX, pY+fY
                HbyA = HbyA - (rAU-rAtU)*gradP
            div0 = fvc_divFlux(pX, pY)
            # The moving wall enters the algorithm here, and nowhere else: the
            # corrected flux is made to satisfy div(phi) = -massSource instead of 0.
            rhsP = div0 + massSource
            rhsP[solid] = 0.0
            pNew = luP.solve(rhsP.ravel()).reshape(ny, nx)
            fX, fY = fvc_grdFlux(pNew, rAtUV[IU], rAtUV[IV])
            return (HbyA - rAtU*fvc_grd(pNew), pNew, pX-fX, pY-fY, div0)

        # Constant flow rate, by linear superposition
        # ------------------------------------------
        # Only the source term depends on the driving force gx, and the pass above is
        # affine in it. So one pass with gx = 0 and one with gx = 1 (and everything
        # else zeroed) give the exact gx that lands the bulk velocity on its target.
        # The moving-wall terms -- mass source, wall traction, blanking -- are all
        # part of the gx = 0 problem, so the superposition is unaffected by them.
        base = advance(rhs,        VAR[IP], massSrc)
        resp = advance(unitSource, noField, noField)
        gx   = (uBulkTarget-bulkVelocity(base[0][IU]))/bulkVelocity(resp[0][IU])

        VARprev  = VAR.copy()
        VAR[0:2] = base[0] + gx*resp[0]
        VAR[IP]  = base[1] + gx*resp[1]
        phiHbyAx = base[2] + gx*resp[2]
        phiHbyAy = base[3] + gx*resp[3]
        outerResid = np.mean(np.abs(VARprev[0:2]-VAR[0:2]))
        meanDiv0   = np.mean(np.abs(base[4] + gx*resp[4]))

    # div(phi) must now equal -massSrc, not zero
    contErr = fvc_divFlux(phiHbyAx, phiHbyAy) + massSrc
    slip    = (np.max(np.abs(VAR[0:2][:, nearBody] - uWall[:, None]))
               if nearBody.any() else 0.0)
    diag = dict(t=t+dt, body=bodyPosition(t+dt), uWall=uWall, gx=gx,
                uBulk=bulkVelocity(VAR[IU]), force=bodyForce(VAR[IP], VAR[0:2], uWall),
                contMax=np.max(np.abs(contErr)), contSum=contErr.sum(),
                meanDiv0=meanDiv0, outerResid=outerResid,
                alphaOldMin=alphaVOld[fluid].min()/V,
                nSwept=int(((alphaVOld < 0.0) & fluid).sum()), slip=slip, cfl=dt*max(np.max(np.abs(VAR[IU]))/dx,
                                      np.max(np.abs(VAR[IV]))/dy),
                massSrc=massSrc, contErr=contErr)
    return phiHbyAx, phiHbyAy, diag


# Initial condition
# -----------------
t = 0.0
updateGeometry(*bodyPosition(t))
if readRestart and not startNew:
    try:
        rst = np.load('restart_moving.npz')
        VAR[...] = rst['VAR']
        phiHbyAx, phiHbyAy, t = rst['phiX'], rst['phiY'], float(rst['t'])
        updateGeometry(*bodyPosition(t))
    except FileNotFoundError:
        print('No restart field found, starting from rest.')
        startNew = True
if startNew:
    VAR[...] = 0.0
    phiHbyAx, phiHbyAy = interpolatePhi(VAR[IU], VAR[IV])

# Timeloop
history = []
for it in range(nSteps):

    phiHbyAx, phiHbyAy, d = timeStep(t, dt, VAR, phiHbyAx, phiHbyAy)
    t = d['t']
    history.append(d)

    print(f'Step {it}   t = {t:.3f}   body x = {d["body"][0]:.4f}   u_body = {d["uWall"][0]:+.4f}')
    print( '------------------')
    print(f'                     Courant number of this step: {d["cfl"]:.2f}')
    print(f'                               timestep used, dt: {dt:.3f}')
    print(f'    global continuity error: {d["contSum"]:.3e}   (sum of div(phi) + S)')
    print(f'     final local continuity error: {d["contMax"]:.3e}')
    print(f'   initial local continuity error: {d["meanDiv0"]:.3e}')
    print(f'    smallest old fluid fraction / cells below zero: {d["alphaOldMin"]:+.4f} / {d["nSwept"]}')
    print(f'              bulk velocity / target: {d["uBulk"]:.9f} / {uBulkTarget}')
    print(f'                     driving force gx: {d["gx"]:.6e}')
    print(f'       max |u - u_body| at the body: {d["slip"]:.6f}')
    print(f'              force on body (x, y): {d["force"][0]:+.5f}, {d["force"][1]:+.5f}')
    print(f'                 outer-iteration change: {d["outerResid"]:.3e}')
    print(f'')

np.savez('restart_moving.npz', VAR=VAR, phiX=phiHbyAx, phiY=phiHbyAy, t=t)
massSrc = history[-1]['massSrc']
contErr = history[-1]['contErr']
divU    = fvc_divFlux(phiHbyAx, phiHbyAy)

In [ ]:
# Verification of the moving-wall treatment
# -----------------------------------------
# T1  Geometric conservation law / free-stream preservation.
#
#     Fill the domain with a uniform u = u_body and translate the body at that same
#     velocity. The fluid then moves rigidly with the body: no relative motion
#     anywhere, no pressure gradient, no traction. The exact solution is u = u_body
#     for all time, so ANY departure is a discretisation error -- and one test
#     exercises every sign at once: the two-level ddt, the mass source, the moving
#     wall traction, the blanking. Get any of them wrong and the error is O(1).
#
#     It is machine precision from the second outer iteration on, at any body speed
#     and for both wall conditions, because the mass source and the old fluid volume
#     are two forms of the same discrete wall flux. The first outer iteration is not
#     exact: it assembles the convective operator from a flux that still belongs to
#     the previous geometry. This is the concrete reason nOuter = 1 is not enough for
#     a moving body, and the table below is what it costs.
#
# T2  Discrete mass conservation is reported every step by the run loop above:
#     contMax = max |div(phi) + S| and contSum, both at roundoff, with no projection
#     applied anywhere.
#
# T3  Consistency of the swept volume. alpha^n V = alpha^{n+1} V - dt S is a backward
#     Euler integration of d(alpha V)/dt, so it should converge to the geometry at
#     the old time as dt is refined. That it does is what says the exactness of T1
#     was not bought by evolving a volume that has come loose from the body. The
#     comparison is made over cells that are cut in both configurations, since a cell
#     that crosses the sliver threshold in between changes by ALPHA_MIN for reasons
#     that have nothing to do with the time integration.
#
#     The same table sets the step restriction of a moving wall. alpha^n goes
#     negative -- and the row sum of the unsteady plus convective operator, which is
#     exactly alpha^n V/dt, changes sign with it -- once the wall sweeps through more
#     than a wall-adjacent cell holds, dt |u_b| > alpha V/A_wall = 2 dWall. It is a
#     condition on the SMALLEST wet cell, not on the mesh spacing, so ALPHA_MIN is
#     what sets it; the run loop reports how many cells violate it.

def freeStreamTest(uBody, nStep=5, nOuterTest=None):
    """max |u - u_body| over nStep steps of rigid translation, restoring all state."""
    global motion, uTrans, uBulkTarget, nOuter
    keep = (motion, uTrans, uBulkTarget, nOuter, VAR.copy())
    motion, uTrans, uBulkTarget = 'translate', uBody, uBody
    if nOuterTest is not None:
        nOuter = nOuterTest

    tT = 0.0
    updateGeometry(*bodyPosition(tT))
    VAR[IU], VAR[IV], VAR[IP] = uBody, 0.0, 0.0
    pX, pY = interpolatePhi(VAR[IU], VAR[IV])
    err = 0.0
    for _ in range(nStep):
        pX, pY, dg = timeStep(tT, dt, VAR, pX, pY)
        tT  = dg['t']
        err = max(err, np.max(np.abs(VAR[IU][fluid]-uBody)),
                       np.max(np.abs(VAR[IV][fluid])))

    motion, uTrans, uBulkTarget, nOuter = keep[0], keep[1], keep[2], keep[3]
    VAR[...] = keep[4]
    return err


print(f'T1  free-stream preservation, bcType = {bcType}: max |u - u_body| after 5 steps')
print( '    nOuter     u_body = 0.37     u_body = 2.00')
for k in (1, 2, 3, 5):
    print(f'    {k:^6d}     {freeStreamTest(0.37, nOuterTest=k):.3e}         '
          f'{freeStreamTest(2.00, nOuterTest=k):.3e}')

print('\nT3  swept volume against the geometry, over cells cut in both configurations')
print( '    dt        mean |alpha^n - alpha_geom|   cells with alpha^n < 0')
for dtT in (0.4, 0.2, 0.1, 0.05, 0.025, 0.0125):
    updateGeometry(Lx/2, Ly/2)
    aGeom, cutOld = alphaV.copy(), (alpha > 0.0) & (alpha < 1.0)
    updateGeometry(Lx/2+0.37*dtT, Ly/2)
    _, aSwept = ibm_movingWall(np.array([0.37, 0.0]), dtT)
    both = cutOld & (alpha > 0.0) & (alpha < 1.0)
    print(f'    {dtT:6.4f}    {np.abs(aSwept-aGeom)[both].mean()/V:.3e}                    '
          f'{int(((aSwept < 0.0) & fluid).sum())}')

updateGeometry(*bodyPosition(t))     # leave the geometry where the run loop left it

In [ ]:
xcB, ycB = bodyPosition(t)
angle = np.linspace(0, 2*np.pi, 200)
xBody = xcB + R*np.cos(angle)
yBody = ycB + R*np.sin(angle)

plt.figure()
plt.title('Volumes (green), fluxes (black & red) and pressure (colormap)')
xmin, xmax = xcB-1.3*R, xcB+1.3*R
ymin, ymax = ycB-1.3*R, ycB+1.3*R

plt.plot(xBody, yBody, 'y')
plt.pcolormesh(XP, YP, VAR[IP])
for x in xf:
    plt.plot([x, x], [yf[0], yf[-1]], 'g', lw=0.5)
for y in yf:
    plt.plot([xf[0], xf[-1]], [y, y], 'g', lw=0.5)
plt.quiver(xf, yp, phiHbyAx, np.zeros_like(phiHbyAx), pivot='mid', scale=0.5)
plt.quiver(xp, yf, np.zeros_like(phiHbyAy), phiHbyAy, color='r', pivot='mid', scale=0.5)
# The number in each cell is div(phi) + S, which is what has to vanish now: div(phi)
# alone does not, and is largest exactly where the wall is sweeping through.
for ix, x in enumerate(xp):
    for iy, y in enumerate(yp):
        if xmin < x < xmax and ymin < y < ymax:
            plt.text(x, y, f'{contErr[iy, ix]:.1E}', horizontalalignment='center')
plt.colorbar()
plt.xlim([xmin, xmax])
plt.ylim([ymin, ymax])

In [ ]:
plt.figure()
plt.title('Colormap: p, vectors: u')
plt.pcolormesh(XP, YP, VAR[IP])
plt.colorbar()
plt.quiver(XP, YP, VAR[IU], VAR[IV])
plt.plot(xBody, yBody, 'y')

plt.figure()
plt.title('Colormap: u')
plt.pcolormesh(XP, YP, VAR[IU])
plt.colorbar()
plt.plot(xBody, yBody, 'y')

# Velocity in the first fluid cell around the body, RELATIVE to the body: no slip
# should drive this to zero, free slip only its normal component.
uW = history[-1]['uWall']
jj, ii = np.where(nearBody)
th = np.arctan2(YP[jj, ii]-ycB, XP[jj, ii]-xcB)
du = VAR[IU][jj, ii]-uW[0]
dv = VAR[IV][jj, ii]-uW[1]
ut = -np.sin(th)*du + np.cos(th)*dv
un =  np.cos(th)*du + np.sin(th)*dv
corner = ((np.roll(solid, -1, 1) | np.roll(solid, 1, 1)) &
          (np.roll(solid, -1, 0) | np.roll(solid, 1, 0)))[jj, ii]

plt.figure()
plt.title('Velocity relative to the body in the first fluid cell around it')
plt.plot(np.degrees(th[~corner]), ut[~corner], 'o', label='tangential, one axis cut')
plt.plot(np.degrees(th[corner]),  ut[corner],  's', label='tangential, both axes cut')
plt.plot(np.degrees(th), un, '.', color='0.6', label='normal')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('angle around the body [deg]')
plt.ylabel('velocity relative to the body')
plt.legend()

plt.figure()
plt.title('Colormap: div(phi) + S, the continuity residual')
plt.pcolormesh(XP, YP, contErr)
plt.colorbar()
plt.plot(xBody, yBody, 'y')

In [ ]:
# Time histories
# --------------
tt = np.array([d['t']       for d in history])
fx = np.array([d['force'][0] for d in history])
fy = np.array([d['force'][1] for d in history])
ub = np.array([d['uWall'][0] for d in history])

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(7, 8))
ax[0].plot(tt, ub, label='$u_{body}$')
ax[0].plot(tt, [d['uBulk'] for d in history], label='bulk $u$')
ax[0].legend(); ax[0].set_ylabel('velocity'); ax[0].axhline(0, color='k', lw=0.5)

ax[1].plot(tt, fx, label='$F_x$')
ax[1].plot(tt, fy, label='$F_y$')
ax[1].legend(); ax[1].set_ylabel('force on body'); ax[1].axhline(0, color='k', lw=0.5)

ax[2].semilogy(tt, [d['contMax'] for d in history], label=r'max $|\nabla\!\cdot\!\phi + S|$')
ax[2].semilogy(tt, [abs(d['contSum']) for d in history], label=r'$|\sum(\nabla\!\cdot\!\phi + S)|$')
ax[2].legend(); ax[2].set_ylabel('continuity'); ax[2].set_xlabel('t')